# GTEx factor model: CoGAPS (FertigLab)

This notebook:
- loads `output/gtex/df_gtex_fbm_filt.rds` (genes × samples)
- loads `output/gtex/CLAMP_K_gtex.rds` (K)
- runs **CoGAPS** with `nPatterns = K`
- writes **B** = Pattern matrix **P** (LVs/patterns × samples) to `output/gtex/cogaps/gtex_B.csv`

Notes:
- CoGAPS is a Bayesian NMF method that factorizes a nonnegative data matrix into **Amplitude (A)** (gene weights) and **Pattern (P)** (sample weights). The **Pattern (P)** matrix is what you want as `B` (patterns × samples).
- Your `df_gtex_fbm_filt.rds` appears z-scored (can contain negatives). CoGAPS requires nonnegative input, so this notebook **shifts** the matrix to be ≥ 0 if needed. For best comparability, prefer using a truly nonnegative expression matrix (e.g., counts/CPM/TPM) instead of z-scores.
- `K = 412` can be heavy; if needed, lower `nPatterns` first to sanity-check.


In [1]:
library(here)
library(CoGAPS)
library(BiocParallel)

# --- inputs ---
gtex_rds <- here('output/gtex/df_gtex_fbm_filt.rds')
k_rds    <- here('output/gtex/CLAMP_K_gtex.rds')

stopifnot(file.exists(gtex_rds), file.exists(k_rds))

gtex_data <- readRDS(gtex_rds)  # genes x samples
K <- readRDS(k_rds)

dim(gtex_data)
K

stopifnot(is.numeric(K), length(K) == 1)
stopifnot(!is.null(rownames(gtex_data)), !is.null(colnames(gtex_data)))


here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



[1] 21613 17382

[1] 412

In [2]:
# --- make input nonnegative if needed ---
X <- as.matrix(gtex_data)

min_x <- min(X, na.rm = TRUE)
if (min_x < 0) {
  # shift so minimum becomes 0
  X <- X - min_x
}

stopifnot(min(X, na.rm = TRUE) >= 0)

rm(gtex_data)
gc()


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,7128015,380.7,12736285,680.2,8616139,460.2
Vcells,388399390,2963.3,1095871318,8360.9,1139844280,8696.4


In [3]:
# --- CoGAPS parameters ---
set.seed(1)

nPatterns   <- as.integer(K)     # number of latent patterns (LVs)
nIterations <- 5000L             # increase for better convergence
seed        <- 1L

# For large datasets, sparseOptimization can help.
params <- CogapsParams(
  nPatterns = nPatterns,
  nIterations = nIterations,
  seed = seed,
  sparseOptimization = TRUE,
  geneNames = rownames(X),
  sampleNames = colnames(X)
)

params


-- Standard Parameters --
nPatterns            412 
nIterations          5000 
seed                 1 
sparseOptimization   TRUE 

-- Sparsity Parameters --
alpha          0.01 
maxGibbsMass   100 

21613 gene names provided
first gene name: WASH7P 

17382 sample names provided
first sample name: GTEX-1117F-0226-SM-5GZZ7 

In [ ]:
# --- run CoGAPS ---
# Use SerialParam for portability; switch to MulticoreParam(...) if you want parallel.
bp <- SerialParam()

cogaps_res <- CoGAPS(X, params = params, BPPARAM = bp)
cogaps_res


In [ ]:
# --- extract matrices ---
# Pattern matrix P: patterns x samples  (this is your B)
P <- getPatternMatrix(cogaps_res)

# Amplitude matrix A: genes x patterns
A <- getAmplitudeMatrix(cogaps_res)

dim(P)
dim(A)

# name LVs consistently
rownames(P) <- paste0('LV', seq_len(nrow(P)))
colnames(P) <- colnames(X)

rownames(A) <- rownames(X)
colnames(A) <- rownames(P)


In [ ]:
# --- write outputs ---
output_dir <- here('output/gtex/cogaps')
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

# B matrix (LVs x samples)
write.csv(P, file = file.path(output_dir, 'gtex_B.csv'), quote = FALSE)

# (optional) save A and full result
write.csv(A, file = file.path(output_dir, 'gtex_A.csv'), quote = FALSE)
saveRDS(cogaps_res, file = file.path(output_dir, 'cogaps_result.rds'))

file.path(output_dir, 'gtex_B.csv')
